# Translator with a stand-in server — the translate path with no model and no GPU

The app is started with `--llama-url` at a stand-in llama-server on this PC (same endpoints and replies as the real one, nothing behind them), the translator is
started over the bridge, Japanese chat is replayed, and the notebook reads the app's events and what the stand-in received: the translator goes Starting -> Loading Model
(the stand-in says 503 for a few seconds) -> Active; a Japanese line gets its translation and an English one is not sent; the prompt is well formed (and how many literal
`<bos>` it holds is reported); three failed lines in a row restart the translator, and the lines that failed are translated afterwards. It also checks that the health wait leaves no `Polling` lines in the system log, that a restart serves each translated line once, that translation turned on later catches up on the newest few lines only, that a live line said during a slow catch-up is served before its last item, and that a request the server never answers costs one line, not the translator (that part waits for the app's 30 s limit). Needs a `test-env` exe and Node 20+;
run Jupyter as Administrator. The same pipeline without Jupyter: `python -m runbook.run translator-stub`.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "runbook" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find runbook above it
from runbook import bridge, common
from runbook.common import Recorder
from runbook.pipelines.translator_stub import TranslatorStub
rec = Recorder("translator-stub")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")
bridge.ensure_installed()                # npm ci in runbook/bridge, the first time

In [ ]:
EXE = common.ask_exe()          # target/release/resonance-stream.exe of this checkout, unless RUNBOOK_TEXT_LOCAL_EXE says otherwise
TranslatorStub(rec, EXE, common.RUNS).run()

## Report
Paste the output of the next cell to Claude.

In [ ]:
print(rec.report())